# AgentFlow NL?SQL Exploration

This notebook focuses on how an agent can probe the semantic layer through natural language, inspect the generated SQL, and learn the boundaries of the fallback translator.

## Setup

An exploratory notebook still starts from the SDK because that is the interface an agent actually owns. The API key comes from the environment and the client timeout is stretched to accommodate local health checks.

In [1]:
from __future__ import annotations

import json
import os
import socket
import subprocess
import sys
import tempfile
import time
import urllib.request
from pathlib import Path
from pprint import pprint

ROOT = Path.cwd().resolve()
SDK_PATH = ROOT / 'sdk'
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(SDK_PATH), 'sseclient-py'],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SDK_PATH))

from agentflow import AgentFlowClient

BASE_URL = os.environ.get('AGENTFLOW_BASE_URL', 'http://127.0.0.1:8000')
API_KEY = os.environ['AGENTFLOW_API_KEY']
DEMO_ORDER_ID = os.environ.get('AGENTFLOW_DEMO_ORDER_ID', 'ORD-20260404-1001')
DB_PATH = Path(os.environ.get('AGENTFLOW_DUCKDB_PATH', 'agentflow_demo.duckdb'))
client = AgentFlowClient(BASE_URL, api_key=API_KEY, timeout=30.0)

print(f'SDK ready from: {SDK_PATH}')
print(f'API base URL: {BASE_URL}')
print(f'Demo order id: {DEMO_ORDER_ID}')
print(f'DuckDB path: {DB_PATH}')


SDK ready from: D:\DE_project\sdk
API base URL: http://127.0.0.1:45637
Demo order id: ORD-20260410-2810
DuckDB path: C:\Users\uedom\AppData\Local\Temp\agentflow-notebook-demo-v2a4dnf5.duckdb


## Discover The Catalog

Before generating SQL, an agent should know which entities and metrics exist. The catalog is the semantic contract that tells the model what is safe to ask for.

In [2]:
catalog = client.catalog()
print('Entities:', ', '.join(sorted(catalog.entities)))
print('Metrics:', ', '.join(sorted(catalog.metrics)))


Entities: order, product, session, user
Metrics: active_sessions, avg_order_value, conversion_rate, error_rate, order_count, revenue


## Probe Supported Question Shapes

The fallback translator is intentionally narrow. Running a few representative prompts makes it obvious which business questions map cleanly to SQL and how the engine structures the answer payload.

In [3]:
questions = [
    'What is the average order value in the last hour?',
    'Which products are out of stock?',
    'Show user USR-10005',
    'top 5 products by revenue this week',
]

for question in questions:
    result = client.query(question)
    rows = result.answer if isinstance(result.answer, list) else [result.answer]
    print(f'QUESTION: {question}')
    print(f'SQL: {result.sql}')
    print('ROWS:')
    pprint(rows[:3])
    print('METADATA:')
    pprint(result.metadata)
    print('=' * 80)


QUESTION: What is the average order value in the last hour?
SQL: SELECT AVG(total_amount) as avg_order_value FROM orders_v2 WHERE status != 'cancelled' AND created_at >= NOW() - INTERVAL '1 hour'
ROWS:
[{'avg_order_value': 289.2790909090909}]
METADATA:
{'data_freshness_seconds': None, 'execution_time_ms': 2, 'rows_returned': 1}
QUESTION: Which products are out of stock?
SQL: SELECT product_id, name, category, stock_quantity FROM products_current WHERE in_stock = FALSE OR stock_quantity = 0
ROWS:
[{'category': 'electronics',
  'name': 'Bluetooth Speaker',
  'product_id': 'PROD-009',
  'stock_quantity': 126}]
METADATA:
{'data_freshness_seconds': None, 'execution_time_ms': 2, 'rows_returned': 1}
QUESTION: Show user USR-10005
SQL: SELECT * FROM users_enriched WHERE user_id = 'USR-10005'
ROWS:
[]
METADATA:
{'data_freshness_seconds': None, 'execution_time_ms': 1, 'rows_returned': 0}
QUESTION: top 5 products by revenue this week
SQL: SELECT SUM(total_amount) as revenue FROM orders_v2 WHERE st

## See Failure Modes Early

A market-grade notebook should also show where the fallback translator stops. That gives evaluators a realistic picture of what needs an LLM-backed semantic planner instead of a rule-based shortcut.

In [4]:
from agentflow.exceptions import AgentFlowError

try:
    client.query('Which warehouse will run out of stock next Tuesday?')
except AgentFlowError as exc:
    print(type(exc).__name__)
    print(str(exc))


## LLM Mode Note

If `ANTHROPIC_API_KEY` is present, AgentFlow can switch from the rule-based translator to Claude-backed SQL generation. The exact same SDK calls still work; only the semantic planner behind `/v1/query` changes.

In [5]:
print('ANTHROPIC_API_KEY configured:', bool(os.getenv('ANTHROPIC_API_KEY')))
print('Fallback mode still returns generated SQL, row samples, and metadata for inspection.')


ANTHROPIC_API_KEY configured: False
Fallback mode still returns generated SQL, row samples, and metadata for inspection.
